In [ ]:
"""
Python Version: 3.10
Dependencies:
    pandas==2.2.0
    numpy==1.26.0
    scikit-learn==1.3.2
    xgboost==1.7.6
    joblib==1.3.2
    shap==0.44.0

Description:
This script performs defect classification (Good, Balling, Lack of Fusion, Keyholing)
for LPBF-processed alloys using machine learning models such as Decision Tree, Random Forest, XGBoost, Gaussain Process Classifier, Gradient Boosting Classifier
and Support Vector Machine. The workflow includes:
    1. Data loading and preprocessing
    2. Model training with GridSearchCV
    3. Model evaluation and saving
    4. Prediction on new data of each alloy with different combination of laser power and scan speeds with probabilities
    5. SHAP-based feature interpretation
"""

import pandas as pd
import numpy as np
import joblib
import warnings
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import shap
import os

warnings.filterwarnings(action="ignore")

# -------------------------------
# 1. Load dataset for training
# -------------------------------
dataset_path = r"C:\Users\gurra\Downloads\Final_ML_excluding 30 exp points.csv"
dataset = pd.read_csv(dataset_path)

# Features & target
features = [
    'power','speed','thickness','dia','Solidt','Sdensity','Sspheat','Sthercondu',
    'Liquidt','Ldensity','LSpheat','Lthercondu','Lsurfacet','Lviscosity',
    'Lfusion','dsigma','Absorptivity','Lvapor'
]
X = dataset[features]
y = dataset['defect']

# Scale features
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# -------------------------------
# 2. Train Random Forest with GridSearchCV
# -------------------------------
param_grid = {
    "n_estimators": [50, 75, 100, 150],
    "max_depth": [4, 5, 6, 7, None],
    "random_state": [42]
}

rf = RandomForestClassifier()
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

# -------------------------------
# 3. Evaluate model
# -------------------------------
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print("\nTrain Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nConfusion Matrix (Test Set):\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification Report (Test Set):\n", 
      classification_report(y_test, y_test_pred, labels=[0,1,2,3],
                             target_names=['Good','Balling','Lack of fusion','Keyholing']))

# -------------------------------
# 4. Save model & scaler
# -------------------------------
joblib.dump(best_model, "RandomForest_model.pkl")
joblib.dump(scaler, "RandomForest_scaler.pkl")

# -------------------------------
# 5. Predict new alloy dataset
# -------------------------------
new_data_path = r"C:\Users\gurra\Downloads\final 5-800w power speed Al alloy predictions.csv"
new_data = pd.read_csv(new_data_path)

# Scale features using saved scaler
X_new_scaled = scaler.transform(new_data[features])

# Predictions and probabilities
y_new_pred = best_model.predict(X_new_scaled)
y_new_proba = best_model.predict_proba(X_new_scaled)

# Map numeric predictions to labels
class_labels = {0:'Good',1:'Balling',2:'Lack of fusion',3:'Keyholing'}
new_data['Predicted_class'] = y_new_pred
new_data['Predicted_Label'] = new_data['Predicted_class'].map(class_labels)

# Add probability columns
proba_df = pd.DataFrame(
    y_new_proba,
    columns=[f"Prob_{class_labels[i]}" for i in sorted(class_labels.keys())]
)
new_data = pd.concat([new_data, proba_df], axis=1)

# Save predictions with probabilities
output_csv = r"C:\Users\gurra\Downloads\EE30-Al_alloy_RF_predictions.csv"
if os.path.exists(output_csv):
    os.remove(output_csv)
new_data.to_csv(output_csv, index=False)
print(f"\nPredictions saved to: {output_csv}")

# -------------------------------
# 6. SHAP Analysis
# -------------------------------
explainer = shap.TreeExplainer(best_model)
shap_values = explainer(X_new_scaled, check_additivity=False)

# Detect number of classes
n_classes = shap_values.values.shape[2] if shap_values.values.ndim==3 else 1
print(f"\nDetected {n_classes} output classes")

# -------------------------------
# 6a. SHAP Beeswarm plots per class
# -------------------------------
for c in range(n_classes):
    class_name = class_labels.get(c, f"Class {c}")
    print(f"\nGenerating beeswarm for {class_name} ...")
    plt.figure(figsize=(10,6))
    shap.summary_plot(
        shap_values.values[:, :, c] if n_classes>1 else shap_values.values,
        X_new_scaled,
        show=False,
        feature_names=features
    )
    plt.title(f"SHAP Beeswarm – {class_name}", fontsize=16, fontweight='bold')
    plt.xlabel("SHAP values", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    plt.show()

# -------------------------------
# 6b. Combined mean absolute SHAP beeswarm
# -------------------------------
if n_classes > 1:
    print("\nGenerating combined mean absolute SHAP beeswarm plot...")
    mean_abs_shap = np.mean(np.abs(shap_values.values), axis=2)
    plt.figure(figsize=(10,6))
    shap.summary_plot(
        mean_abs_shap,
        X_new_scaled,
        show=False,
        feature_names=features
    )
    plt.title("Combined SHAP Beeswarm - Across All Classes", fontsize=16, fontweight='bold')
    plt.xlabel("Mean |SHAP value|", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    plt.show()

# -------------------------------
# 6c. Stacked horizontal SHAP bar chart
# -------------------------------
if n_classes > 1:
    mean_abs_per_class = np.mean(np.abs(shap_values.values), axis=0).T
    total_importance = np.sum(mean_abs_per_class, axis=0)
    sorted_idx = np.argsort(total_importance)[::-1]
    features_sorted = np.array(features)[sorted_idx]
    mean_abs_sorted = mean_abs_per_class[:, sorted_idx]

    colors = ["#08306b", "#377eb8", "#fbb4ae", "#e41a1c"]
    class_names = [class_labels[i] for i in range(n_classes)]

    bottom = np.zeros(len(features_sorted))
    plt.figure(figsize=(10,6))
    for i, (label, color) in enumerate(zip(class_names, colors[:n_classes])):
        plt.barh(features_sorted, mean_abs_sorted[i], left=bottom, color=color, label=label)
        bottom += mean_abs_sorted[i]

    plt.xlabel("Mean |SHAP value|", fontsize=14)
    plt.title("Stacked Feature Importance Across Classes", fontsize=16, fontweight='bold')
    plt.legend(fontsize=12, title_fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    plt.show()
